# Stage 6 — Advocacy Activation Agent Walkthrough

**Chapter 12 · Framework: MCP + OpenAI Agents SDK**

This notebook walks through the Advocacy Activation Agent step by step:

1. Retrieve NPS scores and customer profiles
2. Analyse community engagement
3. Classify advocacy actions using the scoring framework
4. Execute advocacy workflows (case study, referral, community, reviews)
5. Run the full agent end-to-end

**No API keys required** — all tools default to `USE_MOCK_APIS=true`.

In [ ]:
# File      : advocacy_agent_walkthrough.ipynb
# Stage     : 6 — Advocacy
# Chapter   : 12
# Framework : MCP + OpenAI Agents SDK
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os
import sys
import json

# Ensure project root is importable
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["USE_MOCK_APIS"] = "true"
print(f"Project root: {project_root}")
print(f"USE_MOCK_APIS: {os.environ['USE_MOCK_APIS']}")

## 1. NPS Tools — Retrieve Customer Scores

In [ ]:
from stage6_advocacy.tools.nps_tools import (
    get_nps_score,
    get_nps_distribution,
    identify_advocates,
    send_nps_followup,
)

# Retrieve NPS for our three demo customers
for cid in ["CUST-001", "CUST-002", "CUST-003"]:
    result = json.loads(get_nps_score(cid))
    print(f"{cid}: NPS={result['nps_score']}, LTV=${result['ltv']:,.0f}, "
          f"Segment={result['segment']}")

In [ ]:
# NPS distribution across all customers
distribution = json.loads(get_nps_distribution())
print(json.dumps(distribution, indent=2))

In [ ]:
# Identify advocates with NPS >= 9 and LTV >= $40,000
advocates = json.loads(identify_advocates(min_nps=9, min_ltv=40000.0))
print(f"Found {advocates['count']} advocates:")
for a in advocates["advocates"]:
    print(f"  - {a['customer_id']}: {a['contact_email']} "
          f"(NPS={a['nps_score']}, LTV=${a['ltv']:,.0f})")

## 2. Community Activity — Engagement Scores

In [ ]:
from stage6_advocacy.tools.community_invite_tools import (
    get_community_activity,
    invite_to_community,
    request_case_study_participation,
)

for cid in ["CUST-001", "CUST-002", "CUST-003"]:
    activity = json.loads(get_community_activity(cid))
    print(f"{cid}: engagement={activity.get('engagement_score', 0)}/100, "
          f"posts={activity.get('posts_last_90_days', 0)}, "
          f"moderator={activity.get('is_moderator', False)}")

## 3. Scoring Framework — Classify Actions

In [ ]:
from stage6_advocacy.agents.advocacy_activation_agent import classify_advocacy_action

# Build profiles and classify
for cid in ["CUST-001", "CUST-002", "CUST-003", "CUST-004"]:
    profile = json.loads(get_nps_score(cid))
    if "error" in profile:
        print(f"{cid}: {profile['error']}")
        continue
    actions = classify_advocacy_action(profile)
    print(f"{cid} (NPS={profile['nps_score']}, LTV=${profile['ltv']:,.0f}): "
          f"{actions}")

## 4. Review Request Tools — G2 with Cooldown

In [ ]:
from stage6_advocacy.tools.review_request_tools import (
    check_review_request_cooldown,
    request_g2_review,
    get_review_request_history,
)

# Check cooldown for each customer
emails = ["sarah@techcorp.com", "james@growthio.com", "mei@communityplus.org"]
for email in emails:
    cooldown = json.loads(check_review_request_cooldown(email))
    print(f"{email}: in_cooldown={cooldown['in_cooldown']}")

In [ ]:
# Request a G2 review for Mei (not in cooldown)
result = json.loads(request_g2_review(
    "mei@communityplus.org",
    "Your insights on community features would help others discover our platform!"
))
print(json.dumps(result, indent=2))

## 5. Referral Programme Tools

In [ ]:
from stage6_advocacy.tools.referral_programme_tools import (
    trigger_referral_programme,
    get_referral_pipeline,
    create_referral_link,
)

# Enrol CUST-002 (recent expander) in silver tier
enrolment = json.loads(trigger_referral_programme("CUST-002", "silver"))
print("Enrolment:")
print(json.dumps(enrolment, indent=2))

In [ ]:
# Check existing pipeline for CUST-001 (already enrolled)
pipeline = json.loads(get_referral_pipeline("CUST-001"))
print(f"CUST-001 pipeline: {pipeline['total_referrals']} referrals, "
      f"{pipeline['converted']} converted, "
      f"${pipeline['pipeline_value']:,.0f} value")

In [ ]:
# Generate referral link for CUST-002
link = json.loads(create_referral_link("CUST-002"))
print(f"Referral link: {link['referral_link']} (new={link['is_new']})")

## 6. Community Invites & Case Studies

In [ ]:
# Invite CUST-003 to moderator role (already active)
invite = json.loads(invite_to_community("mei@communityplus.org", "moderator"))
print(f"Invite status: {invite['status']}")
print(f"Join link: {invite['join_link']}")

In [ ]:
# Request case study from CUST-001 (high NPS + high LTV)
case_study = json.loads(request_case_study_participation(
    "CUST-001",
    "Pipeline transformation with AI-powered marketing automation"
))
print(f"Case study request: {case_study['status']}")
print(f"Incentive: {case_study['incentive']}")
print("Next steps:")
for step in case_study["next_steps"]:
    print(f"  - {step}")

## 7. NPS Follow-ups

In [ ]:
# Send follow-ups for each score band
followup_cases = [
    ("CUST-001", 10),  # Promoter
    ("CUST-002", 8),   # Passive
    ("CUST-004", 6),   # Detractor
]

for cid, score in followup_cases:
    result = json.loads(send_nps_followup(cid, score))
    print(f"{cid} (NPS={score}): action={result['action']}")
    print(f"  Message: {result['message']}")
    print()

## 8. Full Agent Run (requires OPENAI_API_KEY)

Set `OPENAI_API_KEY` in your environment to run this cell.
Otherwise, use the local tool demo above.

In [ ]:
import asyncio

if os.getenv("OPENAI_API_KEY"):
    from stage6_advocacy.agents.advocacy_activation_agent import run_advocacy_agent

    prompt = """
    Process customer CUST-001 for advocacy activation.
    Retrieve their NPS and community data, classify actions,
    execute each one, and provide a summary.
    """

    result = await run_advocacy_agent(prompt)
    print(result)
else:
    print("OPENAI_API_KEY not set. Skipping agent run.")
    print("All tools were demonstrated individually in the cells above.")

---

**End of walkthrough.** See `project_advocacy_agent/main.py` for the
full three-customer demo script.